In [ ]:
from __future__ import annotations
import logfire
import logging
logging.basicConfig(handlers=[logfire.LogfireLoggingHandler()], level=logging.INFO)

from enum import Enum
import os
import time
import numpy as np
from pathlib import Path
import json
import time
from pydantic import Field
from pydantic_settings import SettingsConfigDict
from wandb import Api

from mmm.settings import mtl_settings as s
from mmm.interactive import configs, training, api
from mmm.api.WorkerState import ws
from mmm.api.functions.compression import CacheSubjects, CacheInstances
from mmm.api.functions.deeplearning import Finetune, Predict
from mmm.api.functions.metrics import Metric

try:
    REGISTRY = os.getenv("MMMEVAL_REGISTRY", "wandb-registry-Universal Evaluation")
    collections = [coll for coll in Api().artifact_type("m3-api-data", REGISTRY).collections()]
    artifact_options = [f"{c.project}/{c.name}:latest" for c in collections]
except Exception as e:
    logfire.warning("Could not fetch artifact options from W&B: {error}", error=e)
    artifact_options = ["wandb-registry-M3Testdata/pets:latest"]

class EvaluationSettings(configs.ExperimentHyperParameters):
    model_config = SettingsConfigDict(env_prefix="MMMEVAL_", title="Evaluation Settings")
    
    # General
    evaluation_id: str = Field(default_factory=lambda: f"evaluation_{int(time.time())}")
    
    artifact_name: str = Field(
        artifact_options[0],
        description="Artifact containing the task-definition.json and other files. Uses :latest by default.",
        examples=artifact_options
    )
    compress_type: api.CompressType | None = Field(None, description="Inferred from task's labels")
    only_labels: None | list[str] = None
    resumable: bool = True
    only_split: None | list[str] = Field(None, description="List of split identifiers to evaluate.", examples=[
        ["meta_official_split"],
        ["kfold_0", "kfold_1"],
    ])

    # Subject defaults
    wsi_max_instances: int | None = 100

    # Instance compression
    instance_compression_workers: int = 1
    instance_compression_batchsize: int = 8

    # Evaluation
    # num_sample_predictions: int = 3
    batch_size: tuple[int, int] | None = None
    num_workers: int = 1

    # Model
    finetune_config: api.FineTuner.Config = api.FineTuner.Config(
        max_epochs=15,
        mtl_train_loop=training.TrainLoopConfig(
            max_steps=100,
            task_sampler=training.CyclicTaskSampler.Config(mode="infinite"),
            log_args=training.LoopLogConfig(progress_bar=False),
        ),
        mtl_val_loop=training.ValLoopConfig(max_steps=-1, log_args=training.LoopLogConfig(progress_bar=False)),
        mtl_train_selector=training.RecurringEventSelector(
            every_n=1, starting_at=1
        ),
        early_stopping=None,
    )
    loops_per_task: int = 5

DB_PREFIX = "evaluation:"

class EvaluationStage(Enum):
    
    COMPRESSING = 0
    TRAINING = 10
    PREDICTING = 20
    EVALUATING = 30
    DONE = 40

    @staticmethod
    def get_stage(eval_id: str) -> EvaluationStage:
        value = s.kv.lindex(f"{DB_PREFIX}stage:{eval_id}", -1)
        if value is None:
            return min(EvaluationStage, key=lambda x: x.value)
        else:
            return EvaluationStage(int(value.decode()))

    @staticmethod
    def set_stage(eval_id: str, new_stage: EvaluationStage):
        logfire.info(
            "Setting evaluation {eval_id} stage to {new_stage}",
            eval_id=eval_id,
            new_stage=new_stage
        )
        assert (current_stage := EvaluationStage.get_stage(eval_id).value) <= new_stage.value, \
            f"{new_stage} is lower than {current_stage}!"
        s.kv.rpush(f"{DB_PREFIX}stage:{eval_id}", new_stage.value)

EvaluationSettings.update_schema(env := configs.EnvByConvention(env_name="evaluation", job_config_folder="."))

In [ ]:
ES = EvaluationSettings.load_config(env)
ES.model_dump(exclude_defaults=True)

## Loading subjects and preparing evaluations

In [ ]:
logfire.configure(environment="evaluation", service_name=ES.evaluation_id)

if (new_stage := EvaluationStage.get_stage(ES.evaluation_id).value) < EvaluationStage.DONE.value:
    with logfire.span("Preparing evaluation {eval_name}", eval_name=ES.evaluation_id) as span:
        # Models read environment variables for some settings that we want to set with config
        os.environ["gigapixelimage_max_instances"] = str(ES.wsi_max_instances)
        span.set_attribute("data_dir", data_dir := Path(Api().artifact(ES.artifact_name).download()))
        task_definition = api.TaskDefinition(**json.loads((data_dir / "task-definition.json").read_text()))
        task_definition.make_relative_paths_absolute(data_dir)
        span.set_attribute("task_definition", task_definition.model_dump(exclude_defaults=True))

        has_dense_label = set(task_definition.get_labeling_config().get_all_labeltypes()).intersection({"volume3dmask", "brushlabels"})
        suggested_compress_type = api.CompressType.rgbimage if has_dense_label else api.CompressType.token
        if ES.compress_type is None:
            ES.compress_type = suggested_compress_type
        elif ES.compress_type != suggested_compress_type:
            logfire.warning(
                "The chosen compress_type {chosen} may be suboptimal for the task_definition, "
                "consider using {suggested} instead.",
                chosen=ES.compress_type,
                suggested=suggested_compress_type,
            )
        
        if ES.batch_size is None:
            ES.batch_size = (8, 8) if ES.compress_type == api.CompressType.rgbimage else (64, 64)
            logfire.info("Set batch_size to {batch_size}", batch_size=ES.batch_size)
        span.set_attribute("compress_type", ES.compress_type)
        
        all_labels = ES.only_labels if ES.only_labels else list(task_definition.get_labeling_config().get_parsed().keys())
        span.set_attribute(
            "labels_to_evaluate",
            all_labels
        )

        all_subject_keys: list[bytes] = CacheSubjects.invoke(
            CacheSubjects.Args(subjects=task_definition.get_subjects(), overwrite=False), ws, s.kv
        ).subject_keys
        all_subject_ids = np.array(list(map(lambda id: id.decode()[len(s.subj_prefix) + 1:], all_subject_keys)))

        evaluations = [
            {
                # "compressed_ds": compressed_ds,
                "train_ids": train_ids,
                "val_ids": val_ids,
                "test_ids": test_ids,
                # "for_label": for_label,
                "split_identifier": split_identifier,
            }
            # for for_label in DS.labels
            for split_identifier, train_ids, val_ids, test_ids in task_definition.generate_splits()
        ]
        if ES.only_split is not None:
            evaluations = [e for e in evaluations if e["split_identifier"] in ES.only_split]
        logfire.info(
            "Prepared data with {num_subjects} subjects with splits: {d}",
            num_subjects=len(all_subject_ids),
            d={e["split_identifier"]: (len(e["train_ids"]), len(e["val_ids"]) if e["val_ids"] is not None else None, len(e["test_ids"])) for e in evaluations},
        )
else:
    logfire.info(
        "Evaluation {evaluation_id} already completed: {stage}",
        evaluation_id=ES.evaluation_id,
        stage=EvaluationStage(new_stage)
    )


## Precomputing neural representations

In [ ]:
if EvaluationStage.get_stage(ES.evaluation_id).value <= EvaluationStage.COMPRESSING.value:
    CacheInstances.invoke(
        CacheInstances.Args(
            for_type=ES.compress_type,
            subject_keys=all_subject_keys,
            with_labels=all_labels,
            batch_size=ES.instance_compression_batchsize,
            num_workers=ES.instance_compression_workers,
            skip_if_exists=True,
        ),
        ws, 
        s.kv)
    EvaluationStage.set_stage(ES.evaluation_id, EvaluationStage.TRAINING)

# Fine-tuning


In [ ]:
def build_train_args_kwargs(split_identifier, train_ids, val_ids, test_ids):
    s.kv.delete(train_dataset_key := f"datasets:{ES.evaluation_id}_{split_identifier}_train")
    s.kv.delete(test_dataset_key := f"datasets:{ES.evaluation_id}_{split_identifier}_test")

    s.kv.sadd(train_dataset_key, *all_subject_ids[train_ids])
    s.kv.sadd(test_dataset_key, *all_subject_ids[test_ids])

    if val_ids is not None:
        s.kv.delete(val_dataset_key := f"datasets:{ES.evaluation_id}_{split_identifier}_val")
        s.kv.sadd(val_dataset_key, *all_subject_ids[val_ids])
        
    cohorts = []
    for label in all_labels:
        cfg = task_definition.get_labeling_config().get_parsed()[label]
        assert (
            len(cfg["to_name"]) == 1
        ), f"Only one input allowed for now: {cfg['to_name']}"
        label_cohort = api.KVReprCohort.Config(
            batch_size=ES.batch_size,
            num_workers=ES.num_workers,
            compress_type=ES.compress_type,
            labeling_config=task_definition.get_labeling_config(),
            for_data=api.ReprDataset.Config(
                for_input=cfg["to_name"][0], for_label=label
            ),
            train_dataset=train_dataset_key,
            validation_dataset=(
                None if val_ids is None else val_dataset_key
            ),
        )
        cohorts.append(label_cohort)
    
    run_wandb_args = dict(
        project=ES.wandb_project,
        group=ES.evaluation_id,
        tags=task_definition.meta.tags,
        config={
            "config": ES.model_dump_for_wandb(),
            "evaluation": task_definition.model_dump(),
        },
        resume=ES.resumable,
        name=f"{split_identifier}",
        id=f"{ES.evaluation_id}_{split_identifier}"
    )

    return Finetune.Args(
        cohorts=cohorts,
        finetuning_id=run_wandb_args["id"],
        cfg=ES.finetune_config,
        wandb_args=run_wandb_args,
        wandb_artifacts=[ES.artifact_name],
        num_loops=ES.loops_per_task,
        lock_for_distributed=False,
        eventqueue=run_wandb_args["id"],
    )

train_configs = []
for e in list(evaluations):
    split_identifier, train_ids, val_ids, test_ids = (
        e["split_identifier"],
        e["train_ids"],
        e["val_ids"],
        e["test_ids"],
    )
    train_configs.append(
        build_train_args_kwargs(split_identifier, train_ids, val_ids, test_ids)
    )
all_train_configs = train_configs.copy()
if EvaluationStage.get_stage(ES.evaluation_id).value <= EvaluationStage.TRAINING.value:

    train_index = 0
    while len(train_configs) > 0:
        train_args = train_configs[train_index]
        train_result: Finetune.Results = Finetune.invoke(train_args, ws, s.kv)
        if train_result.state == "done":
            print(f"Finished training {train_args.model_id}")
            train_configs.pop(train_index)
        else:
            train_index = (train_index + 1) % len(train_configs)
    EvaluationStage.set_stage(ES.evaluation_id, EvaluationStage.PREDICTING)


## Adding predictions to all subjects

In [ ]:
if EvaluationStage.get_stage(ES.evaluation_id).value <= EvaluationStage.PREDICTING.value:
    # by this point all training are finished, start testing latest and best models
    for evaluation, train_config in zip(evaluations, all_train_configs):
        test_indices = evaluation["test_ids"]
        for checkpoint in ["latest", "bestbyvalidation"]:
            with logfire.span(
                "Predicting with {model_id}, checkpoint {checkpoint} on split {split_id}",
                model_id=train_config.model_id,
                checkpoint=checkpoint,
                split_id=evaluation["split_identifier"],
            ):
                Predict.invoke(
                    Predict.Args(
                        model_id=train_config.model_id,
                        subjects=[all_subject_keys[test_index] for test_index in test_indices],
                        label_config=task_definition.get_labeling_config(),
                        commit_to_subject_key=True,
                        checkpoint=checkpoint,
                        compress_type=ES.compress_type,
                        compress_batchsize=32,
                        compress_num_workers=2,
                    ),
                    ws,
                    s.kv
                )
    EvaluationStage.set_stage(ES.evaluation_id, EvaluationStage.EVALUATING)


## Computing metrics

In [ ]:
if EvaluationStage.get_stage(ES.evaluation_id).value <= EvaluationStage.EVALUATING.value:
    for evaluation, train_config in zip(evaluations, all_train_configs):
        split_identifier = evaluation["split_identifier"]
        with logfire.span("Computing test metrics for split {split_name}", split_name=split_identifier):
            metric_result: Metric.Results = Metric.invoke(
                Metric.Args(
                    dataset=f"datasets:{ES.evaluation_id}_{split_identifier}_test",
                    to_wandb=train_config.wandb_args,
                    for_models=[
                        f"{train_config.model_id}|{model_version}" for model_version in ["latest", "bestbyvalidation"]
                    ],
                    label_config=task_definition.get_labeling_config(),
                ),
                ws,
                s.kv,
            )

    EvaluationStage.set_stage(ES.evaluation_id, EvaluationStage.DONE)
